Double-checking one sample from the Pittsburgh .json file.
If the key is just "start_point" (without the rvs_ prefix), this script will crash.
If they match the Manhattan format, we are good to go.

In [1]:
import os
import json
import pandas as pd

# Use relative pathing to go up from 'notebooks/' to the root
# then down into 'data/'
base_data_dir = os.path.join("..", "data")

cities = ["pittsburgh", "philadelphia", "manhattan"]

for city in cities:
    print(f"\n🔍 Inspecting Schema for: {city.upper()}")
    
    # RVS files are often named differently, let's look for ANY json in that folder
    city_dir = os.path.join(base_data_dir, city)
    
    if not os.path.exists(city_dir):
        print(f"❌ City directory not found at: {city_dir}")
        continue
    
    # Find the first .json or .jsonl file in the directory
    json_files = [f for f in os.listdir(city_dir) if f.endswith(('.json', '.jsonl'))]
    
    if not json_files:
        print(f"⚠️ No JSON files found in {city_dir}. Check your download/unzip step.")
        continue
        
    json_path = os.path.join(city_dir, json_files[0])
    print(f"📂 Found file: {json_path}")

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            # RVS JSONs can be massive; just peek at the first object
            # If it's a standard JSON list, we might need a different peek strategy
            content = f.read(1000) # Read first 1000 chars
            
            # Check if it's JSONL (one object per line) or standard JSON (starts with [)
            f.seek(0)
            if content.strip().startswith('['):
                # Standard JSON Array
                data = json.load(f)
                sample = data[0]
            else:
                # JSONL / Trailing Data format
                first_line = f.readline()
                sample = json.loads(first_line)
            
        keys = list(sample.keys())
        print(f"✅ Keys found: {keys}")
        
        # The specific check for your batch_labeling.py
        target_keys = ['rvs_start_point', 'rvs_goal_point']
        if all(k in keys for k in target_keys):
            print("💎 Format Match: 'rvs_' prefix is present.")
        elif 'start_point' in keys:
            print("⚠️ Format Mismatch: Found 'start_point' instead of 'rvs_start_point'.")
        else:
            print("❓ Unknown Format: Check the printed keys above.")

    except Exception as e:
        print(f"❌ Error reading {json_path}: {e}")


🔍 Inspecting Schema for: PITTSBURGH
📂 Found file: ..\data\pittsburgh\pittsburgh.json
✅ Keys found: ['rvs_sample_number', 'content', 'rvs_path', 'rvs_goal_point', 'key', 'region', 'rvs_start_point', 'landmarks']
💎 Format Match: 'rvs_' prefix is present.

🔍 Inspecting Schema for: PHILADELPHIA
📂 Found file: ..\data\philadelphia\philadelphia.json
✅ Keys found: ['content', 'rvs_goal_point', 'key', 'region', 'rvs_start_point']
💎 Format Match: 'rvs_' prefix is present.

🔍 Inspecting Schema for: MANHATTAN
📂 Found file: ..\data\manhattan\manhattan.json
✅ Keys found: ['rvs_sample_number', 'content', 'rvs_path', 'rvs_goal_point', 'key', 'region', 'rvs_start_point', 'landmarks']
💎 Format Match: 'rvs_' prefix is present.


Batch labeling philadelphia resulted in:
📊 Label Distribution:
oracle_label
- Ambiguous        930
- Contradictory    284
- Answerable        64
So we audit it:

In [2]:
import pandas as pd
df_phl = pd.read_parquet("../data/philadelphia/philadelphia_silver_standard.parquet")

# Look at the top 5 Ambiguous samples
ambiguous_samples = df_phl[df_phl['oracle_label'] == 'Ambiguous'].head(5)

for _, row in ambiguous_samples.iterrows():
    print(f"ID: {row['sample_id']} | Instruction: {row['instruction']}")
    print(f"Found {row['candidate_count']} candidates for noun: '{row['extracted_noun']}'")
    print("-" * 30)

ID: N/A | Instruction: Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street, on the block with a Cinemark cinema. An Acme supermarket is north on the next block. 
Found 3 candidates for noun: 'None'
------------------------------
ID: N/A | Instruction: Meet me at the cafe north of you on the north side of West Girard Avenue. BB&T bank is southwest of me and an ice cream shop is on my east.

Found 47 candidates for noun: 'None'
------------------------------
ID: N/A | Instruction: Meet me at the historic memorial on the south side of Arch Street. It is a few steps west of the grave yard.
Found 82 candidates for noun: 'None'
------------------------------
ID: N/A | Instruction: Go south and a bit east. You'll find me at the cafe across the street from a bank. The cafe is right on the corner of Arch Street, south of Drexel University College of Nursing and Health Professions.
Found 25 candidates for noun: 'None'
------------------------------
ID: N/A | Instruction: I a

In [3]:
import re

def heuristic_extract(text):
    # Pattern: "at [Name] [optional: on/near Street]"
    # This specifically looks for words following "at" or "the" before a street or comma
    match = re.search(r"(?:at|to|me at|is at) ([\w\s&']+?)(?: on| near| across| south| north| west| east|,|\.)", text)
    if match:
        return match.group(1).strip()
    return None

# Test on the audit samples
samples = [
    "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street...",
    "I am at the American Eagle Outfitters which is in the middle of the block..."
]

for s in samples:
    print(f"Text: {s[:40]}... -> Extracted: {heuristic_extract(s)}")

Text: Meet to the west of you, at Ben & Jerry'... -> Extracted: the
Text: I am at the American Eagle Outfitters wh... -> Extracted: the American Eagle Outfitters which is in the middle of the block


One is too lazy and the other is too greedy.

In [4]:
import re

def clean_landmark(text):
    # Remove common leading noise
    text = re.sub(r'^(the|a|an|at|to|me at|is at)\s+', '', text, flags=re.IGNORECASE)
    # Cut off at the first spatial preposition or relative clause
    # This acts as a "Stop Word" list for landmark names
    stops = [
        r'\s+on\s+', r'\s+at\s+', r'\s+near\s+', r'\s+across\s+', 
        r'\s+which\s+', r'\s+is\s+', r'\s+south\s+', r'\s+north\s+', 
        r'\s+west\s+', r'\s+east\s+', r',', r'\.'
    ]
    for stop in stops:
        text = re.split(stop, text, flags=re.IGNORECASE)[0]
    return text.strip()

def extract_rvs_target_v2(text):
    # 1. Identify the "Target Zone" (usually right after 'at' or 'to')
    # We look for the most likely starting point of the goal landmark
    match = re.search(r"(?:at|to|me at|is at)\s+(.*)", text, re.IGNORECASE)
    if match:
        raw_zone = match.group(1)
        return clean_landmark(raw_zone)
    return None

# --- TEST DRIVE ---
test_samples = [
    "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street...",
    "I am at the American Eagle Outfitters which is in the middle of the block...",
    "Meet me at the historic memorial on the south side of Arch Street."
]

for s in test_samples:
    print(f"Original: {s[:50]}...")
    print(f"Extracted: [{extract_rvs_target_v2(s)}]")
    print("-" * 10)

Original: Meet to the west of you, at Ben & Jerry's ice crea...
Extracted: [west of you]
----------
Original: I am at the American Eagle Outfitters which is in ...
Extracted: [American Eagle Outfitters]
----------
Original: Meet me at the historic memorial on the south side...
Extracted: [historic memorial]
----------


In [5]:
import re

def clean_landmark_v3(text):
    # 1. Skip Relative Directions at the start (e.g., "west of you, at...")
    text = re.sub(r'^(to the\s+\w+\s+of\s+\w+[,]?\s+)', '', text, flags=re.IGNORECASE)
    
    # 2. Standard leading noise removal
    text = re.sub(r'^(the|a|an|at|to|me at|is at)\s+', '', text, flags=re.IGNORECASE)
    
    # 3. Stop Word clipping
    stops = [
        r'\s+on\s+', r'\s+at\s+', r'\s+near\s+', r'\s+across\s+', 
        r'\s+which\s+', r'\s+is\s+', r'\s+south\s+', r'\s+north\s+', 
        r'\s+west\s+', r'\s+east\s+', r',', r'\.'
    ]
    for stop in stops:
        text = re.split(stop, text, flags=re.IGNORECASE)[0]
    
    return text.strip()

def extract_rvs_target_v3(text):
    # Search for the anchor that likely starts the goal
    # We prioritize 'at' because 'to' is often used for directions (e.g. 'to the west')
    match = re.search(r"(?:at|me at|is at)\s+(.*)", text, re.IGNORECASE)
    if not match:
        # Fallback to 'to' if 'at' isn't found
        match = re.search(r"(?:to)\s+(.*)", text, re.IGNORECASE)
        
    if match:
        raw_zone = match.group(1)
        return clean_landmark_v3(raw_zone)
    return None

# --- TEST DRIVE ---
s1 = "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street..."
print(f"Extracted: [{extract_rvs_target_v3(s1)}]")

Extracted: [Ben & Jerry's ice cream]


🧪 Cell 1: The "Cross-City" Sanity Check

In [12]:
import sys
import os
import re

# Get the absolute path to the project root (one level up from 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

print(f"✅ Added to path: {project_root}")

✅ Added to path: c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning


In [42]:
%reload_ext autoreload
%autoreload 2
from src.extraction_utils import extract_rvs_target

In [43]:
# 1. Define the failure cases from the previous audit
test_cases = [
    "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street.",
    "I am at the American Eagle Outfitters which is in the middle of the block.",
    "Meet me at the historic memorial on the south side of Arch Street.",
    "Go to the Starbucks on the corner."
]

print(f"{'INPUT TEXT':<50} | {'EXTRACTED NOUN':<25} | {'CATEGORY'}")
print("-" * 90)

# 2. Execute extraction
for text in test_cases:
    category, noun = extract_rvs_target(text)
    print(f"{text[:48]:<50} | {str(noun):<25} | {category}")

# 3. Logic Check
if "Ben & Jerry's" in str(test_cases[0]) and "Ben & Jerry's" not in str(extract_rvs_target(test_cases[0])[1]):
    print("\n❌ STILL CLIPPING: The 'none' or 'partial' bug is still active.")
else:
    print("\n✅ SUCCESS: Full brand spans are being captured.")

INPUT TEXT                                         | EXTRACTED NOUN            | CATEGORY
------------------------------------------------------------------------------------------
Meet to the west of you, at Ben & Jerry's ice cr   | Ben & Jerry's ice cream   | UNKNOWN
I am at the American Eagle Outfitters which is i   | American Eagle Outfitters | UNKNOWN
Meet me at the historic memorial on the south si   | historic memorial         | MONUMENT
Go to the Starbucks on the corner.                 | Starbucks                 | UNKNOWN

✅ SUCCESS: Full brand spans are being captured.


In [44]:
from src.extraction_utils import extract_rvs_target

# Test cases representing different city data styles
cross_city_tests = [
    # Philadelphia (The multi-word brand problem)
    "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street.",
    # Manhattan (The standard 'at' anchor)
    "Go to the Starbucks at the corner of 42nd and Broadway.",
    # Pittsburgh (The 'near' landmark style)
    "I am waiting at the historic memorial near the park entrance.",
    # Edge case: No 'at', just 'to'
    "Walk to the post office."
]

print("🔍 RUNNING CROSS-CITY EXTRACTION TEST\n" + "="*40)
for i, text in enumerate(cross_city_tests):
    cat, noun = extract_rvs_target(text)
    print(f"Sample {i+1}: {text[:50]}...")
    print(f"  Result -> Category: [{cat}], Noun: [{noun}]")
    print("-" * 40)

🔍 RUNNING CROSS-CITY EXTRACTION TEST
Sample 1: Meet to the west of you, at Ben & Jerry's ice crea...
  Result -> Category: [UNKNOWN], Noun: [Ben & Jerry's ice cream]
----------------------------------------
Sample 2: Go to the Starbucks at the corner of 42nd and Broa...
  Result -> Category: [UNKNOWN], Noun: [Starbucks]
----------------------------------------
Sample 3: I am waiting at the historic memorial near the par...
  Result -> Category: [MONUMENT], Noun: [historic memorial]
----------------------------------------
Sample 4: Walk to the post office....
  Result -> Category: [POST], Noun: [post office]
----------------------------------------


In [45]:
from src.extraction_utils import extract_rvs_target

test_brands = [
    "Meet at Ben & Jerry's ice cream",
    "Go to the Starbucks on 4th",
    "I am at American Eagle Outfitters"
]

print("Check: Is the 'Brand Path' preserved?")
for text in test_brands:
    cat, noun = extract_rvs_target(text)
    print(f"Input: {text}")
    print(f"  -> Extracted Noun (for Precision Search): [{noun}]")
    print(f"  -> Resolved Category (for Broad Filter): [{cat}]")
    print("-" * 30)

Check: Is the 'Brand Path' preserved?
Input: Meet at Ben & Jerry's ice cream
  -> Extracted Noun (for Precision Search): [Ben & Jerry's ice cream]
  -> Resolved Category (for Broad Filter): [UNKNOWN]
------------------------------
Input: Go to the Starbucks on 4th
  -> Extracted Noun (for Precision Search): [Starbucks]
  -> Resolved Category (for Broad Filter): [UNKNOWN]
------------------------------
Input: I am at American Eagle Outfitters
  -> Extracted Noun (for Precision Search): [American Eagle Outfitters]
  -> Resolved Category (for Broad Filter): [UNKNOWN]
------------------------------


Making sure the fix didn't break already existing logic:

In [46]:
# Cell 1: The Multi-City Regression Suite

from src.extraction_utils import extract_rvs_target

# Regression Test Cases
regression_tests = {
    "Philadelphia (Brands)": "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street.",
    "Manhattan (Intersections)": "Go to the Starbucks at the corner of 4th and Broadway.",
    "Pittsburgh (Descriptive)": "I am waiting at the historic memorial near the park entrance.",
    "Standard (Simple)": "Walk to the post office.",
    "Edge Case (Articles)": "Meet me at the Chase Bank.",
    "Edge Case (Directions)": "To the north of the building, at the entrance."
}

print(f"{'CITY/STYLE':<25} | {'EXTRACTED NOUN':<30} | {'CATEGORY'}")
print("="*85)

for style, text in regression_tests.items():
    cat, noun = extract_rvs_target(text)
    print(f"{style:<25} | {str(noun):<30} | {cat}")

CITY/STYLE                | EXTRACTED NOUN                 | CATEGORY
Philadelphia (Brands)     | Ben & Jerry's ice cream        | UNKNOWN
Manhattan (Intersections) | Starbucks                      | UNKNOWN
Pittsburgh (Descriptive)  | historic memorial              | MONUMENT
Standard (Simple)         | post office                    | POST
Edge Case (Articles)      | Chase Bank                     | BANK
Edge Case (Directions)    | entrance                       | ENTRANCE


In [47]:
# Cell 2: The "None" & "Unknown" Audit
import pandas as pd
import os

cities = ["manhattan", "pittsburgh"]
results = []

for city in cities:
    path = f"../data/{city}/{city}_silver_standard.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        # Apply the new extraction to a sample of the data
        sample_df = df.sample(min(100, len(df)))
        
        # Count how many now result in UNKNOWN vs before
        unknown_count = 0
        for _, row in sample_df.iterrows():
            cat, noun = extract_rvs_target(row['instruction'])
            if cat == "UNKNOWN":
                unknown_count += 1
        
        results.append({"City": city, "Unknown_Rate": f"{(unknown_count/len(sample_df))*100:.1f}%"})

print("🕵️ REGRESSION AUDIT (Sample of 100 rows per city)")
print(pd.DataFrame(results))

🕵️ REGRESSION AUDIT (Sample of 100 rows per city)
         City Unknown_Rate
0   manhattan        13.0%
1  pittsburgh        17.0%


In [51]:
import json
import pandas as pd
import os
import random

philly_path = "../data/philadelphia/philadelphia.json"

if os.path.exists(philly_path):
    all_instructions = []
    
    with open(philly_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                item = json.loads(line)
                
                # Try common RVS keys
                if 'instruction' in item:
                    all_instructions.append(item['instruction'])
                elif 'instructions' in item:
                    # Some files have a list of instructions per landmark
                    val = item['instructions']
                    if isinstance(val, list): all_instructions.extend(val)
                    else: all_instructions.append(val)
                elif 'text' in item:
                    all_instructions.append(item['text'])
                else:
                    # If we don't recognize the key, let's see what's in there
                    if line_num == 1:
                        print(f"DEBUG: First line keys found: {list(item.keys())}")
            except Exception:
                continue

    if not all_instructions:
        print("❌ Could not find any instruction text. Check the 'DEBUG' output above for key names.")
    else:
        # 2. Sample and Run Audit
        sample_size = min(500, len(all_instructions))
        sample_inst = random.sample(all_instructions, sample_size)
        
        unknown_count = 0
        extracted_results = []

        for text in sample_inst:
            cat, noun = extract_rvs_target(text)
            if cat == "UNKNOWN":
                unknown_count += 1
            extracted_results.append({"Noun": noun, "Category": cat})

        philly_rate = (unknown_count / sample_size) * 100
        
        print(f"🔔 PHILADELPHIA FORECAST (N={sample_size})")
        print(f"Unknown Rate: {philly_rate:.1f}%")
        print(f"Categorized: {100 - philly_rate:.1f}%")
        
        print("\n🔍 TOP PHILLY SAMPLES:")
        print(pd.DataFrame(extracted_results).head(15))
else:
    print("❌ Philadelphia file not found.")

DEBUG: First line keys found: ['content', 'rvs_goal_point', 'key', 'region', 'rvs_start_point']
❌ Could not find any instruction text. Check the 'DEBUG' output above for key names.


In [52]:
import json
import pandas as pd
import os
import random

philly_path = "../data/philadelphia/philadelphia.json"

if os.path.exists(philly_path):
    all_instructions = []
    
    with open(philly_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                item = json.loads(line)
                # The 'DEBUG' confirmed the key is 'content'
                if 'content' in item:
                    all_instructions.append(item['content'])
            except Exception:
                continue

    if not all_instructions:
        print("❌ Still no instructions found. Double-check if 'content' is a string.")
    else:
        sample_size = min(500, len(all_instructions))
        sample_inst = random.sample(all_instructions, sample_size)
        
        extracted_results = []
        unknown_count = 0

        for text in sample_inst:
            cat, noun = extract_rvs_target(text)
            if cat == "UNKNOWN":
                unknown_count += 1
            extracted_results.append({"Noun": noun, "Category": cat})

        philly_rate = (unknown_count / sample_size) * 100
        print(f"🔔 PHILADELPHIA FORECAST (Key: 'content')")
        print(f"Unknown Rate: {philly_rate:.1f}%")
        print(f"Categorized: {100 - philly_rate:.1f}%")
        
        print("\n🔍 SAMPLE EXTRACTIONS:")
        print(pd.DataFrame(extracted_results).head(15))
else:
    print("❌ Philadelphia file not found.")

🔔 PHILADELPHIA FORECAST (Key: 'content')
Unknown Rate: 31.4%
Categorized: 68.6%

🔍 SAMPLE EXTRACTIONS:
                                     Noun    Category
0                             gas station     STATION
1                    tennis pitch to your     UNKNOWN
2            historic building and museum    BUILDING
3                         car repair shop        SHOP
4                             parking lot     PARKING
5                       next intersection     UNKNOWN
6          same spot with social facility     UNKNOWN
7                         Wells Fargo atm     UNKNOWN
8                            CVS Pharmacy    PHARMACY
9   7-Eleven by the short Shedwick Street      STREET
10                        bicycle parking     BICYCLE
11         parking ticket vending machine     PARKING
12                                   cafe        CAFE
13                             restaurant  RESTAURANT
14                                Clarion     UNKNOWN


In [48]:
# Cell 3: Visual Validation of the "Anchor Jump"
complex_cases = [
    "To the south of you, at the University of Pennsylvania.",
    "Go to the shop at the end of the street.",
    "Meet at the Stop & Shop."
]

print("🎯 COMPLEX ANCHOR TEST")
print("-" * 50)
for text in complex_cases:
    cat, noun = extract_rvs_target(text)
    print(f"Original: {text}")
    print(f"  -> Noun: [{noun}] | Cat: [{cat}]")
    print("-" * 50)

🎯 COMPLEX ANCHOR TEST
--------------------------------------------------
Original: To the south of you, at the University of Pennsylvania.
  -> Noun: [University of Pennsylvania] | Cat: [SCHOOL]
--------------------------------------------------
Original: Go to the shop at the end of the street.
  -> Noun: [shop] | Cat: [SHOP]
--------------------------------------------------
Original: Meet at the Stop & Shop.
  -> Noun: [Stop & Shop] | Cat: [SHOP]
--------------------------------------------------


Somehow, although extraction seems to work well now, the results got worse lol:
📊 Label Distribution:
oracle_label
- Ambiguous        931
- Contradictory    315
- Answerable        31

In [54]:
import pandas as pd

df_philly = pd.read_parquet("../data/philadelphia/philadelphia_silver_standard.parquet")

# Print all columns to see what the script actually produced
print(f"Columns found in Parquet: {df_philly.columns.tolist()}")

# Attempt to find the right keys (handling 'content' vs 'instruction')
inst_key = 'content' if 'content' in df_philly.columns else 'instruction'

# Filter for Ambiguous and sample
ambig_df = df_philly[df_philly['oracle_label'] == 'Ambiguous']

if not ambig_df.empty:
    sample_size = min(5, len(ambig_df))
    for i, row in ambig_df.sample(sample_size).iterrows():
        print("-" * 50)
        print(f"TEXT: {row.get(inst_key, 'N/A')}")
        # Print the whole row to see where the landmark info went
        print(f"LABEL: {row['oracle_label']}")
        # Look for any column that might contain our landmark
        possible_cols = ['extracted_landmark', 'landmark', 'noun', 'target']
        for col in possible_cols:
            if col in row:
                print(f"EXTRACTED ({col}): {row[col]}")
else:
    print("No Ambiguous rows found to audit.")

Columns found in Parquet: ['sample_id', 'city', 'instruction', 'oracle_label', 'candidate_count', 'start_node', 'gold_goal_node', 'extracted_category', 'extracted_noun', 'target_tags']
--------------------------------------------------
TEXT: Let's grab a bagel. Meet up with me at the restaurant on south 3rd street. A Rite Aid it northeast of it, a block or so. A pub is over a block to the east of it.
LABEL: Ambiguous
--------------------------------------------------
TEXT: Meet me at the parking lot ok? It's west of the South 12th Street. Historic Building is south a block and a half, and is east a bit. There is parking to the north and east, about a half block or so.
LABEL: Ambiguous
--------------------------------------------------
TEXT: Meet me at the school on Reno Street. It's the building that's on the east side of that small block. A french cafe is west of here and the dog park is a block north.
LABEL: Ambiguous
--------------------------------------------------
TEXT: You will 

In [55]:
# Check a single sample
test_row = df_philly.iloc[0]
print(f"Start Node: {test_row['start_node']}")
# Go to Google Maps and paste these coordinates. 
# Are they actually in Philadelphia or in the Atlantic Ocean/Antarctica?

Start Node: #6593007625


In [57]:
import pickle
import networkx as nx
import os

# 1. Load using standard pickle
graph_path = "../data/philadelphia/philadelphia_graph.gpickle"

if os.path.exists(graph_path):
    with open(graph_path, 'rb') as f:
        G = pickle.load(f)
    print(f"✅ Graph loaded! Type: {type(G)}")
    print(f"Total Nodes: {G.number_of_nodes()}")

    # 2. Check for the node from your audit
    # We will test common RVS formatting variations
    target_id = "6593007625"
    variations = [target_id, int(target_id), f"1#{target_id}", f"node/{target_id}"]
    
    found = False
    for v in variations:
        if v in G:
            print(f"🎯 FOUND NODE with key format: {type(v)} -> {v}")
            print(f"Data: {G.nodes[v]}")
            found = True
            break
    
    if not found:
        print("❌ Node NOT found in graph under any common format.")
        print(f"Sample node ID from graph: {list(G.nodes())[0]}")
else:
    print("❌ Path not found.")

C:\Users\adan\AppData\Local\Temp\ipykernel_21804\1546586476.py:10: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


✅ Graph loaded! Type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
Total Nodes: 63737
🎯 FOUND NODE with key format: <class 'str'> -> 1#6593007625
Data: {'highway': 'footway', 'osmid': '1#6593007625', 'x': -75.18328746148863, 'y': 39.954736552352, 'name': 'projected-poi'}


In [58]:
import pandas as pd
import pickle
import networkx as nx
from shapely.geometry import Point
from scipy.spatial import cKDTree

# 1. Load Resources
with open("../data/philadelphia/philadelphia_graph.gpickle", 'rb') as f:
    G = pickle.load(f)
poi_df = pd.read_pickle("../data/philadelphia/philadelphia_poi.pkl")

# 2. Pick a "Failed" Case (Dunkin from earlier)
# Target Node: #6593007625 (Start)
# Target Noun: "Dunkin"
start_node = "1#6593007625"
target_noun = "Dunkin"

# 3. Get Start Point Coords
s_data = G.nodes[start_node]
start_coords = (s_data['x'], s_data['y']) # (Lon, Lat)
print(f"📍 Start Point: {start_coords}")

# 4. Find candidates for "Dunkin"
# We'll do a raw string search in the POI name column
candidates = poi_df[poi_df['name'].str.contains(target_noun, case=False, na=False)]
print(f"🔍 Found {len(candidates)} Dunkin candidates in Philly POI file.")

# 5. Calculate Distances MANUALLY
def get_dist(p1, p2):
    # Rough Euclidean for diagnostic (Degrees to Meters approx)
    return ((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)**0.5 * 111000

print("\n📏 Manual Distance Check:")
for i, row in candidates.head(10).iterrows():
    # Check if 'centroid' or 'geometry' is the right col
    poi_coords = (row['centroid'].x, row['centroid'].y)
    d = get_dist(start_coords, poi_coords)
    print(f"Candidate {i}: Distance = {d:.2f}m | Coords: {poi_coords}")

C:\Users\adan\AppData\Local\Temp\ipykernel_21804\4177925748.py:9: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)
C:\Users\adan\miniconda3\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


📍 Start Point: (-75.18328746148863, 39.954736552352)
🔍 Found 31 Dunkin candidates in Philly POI file.

📏 Manual Distance Check:
Candidate 355: Distance = 2336.33m | Coords: (-75.1623684, 39.95241)
Candidate 371: Distance = 3586.58m | Coords: (-75.1607302, 39.9316021)
Candidate 386: Distance = 5385.93m | Coords: (-75.1562858, 39.9144217)
Candidate 409: Distance = 3791.55m | Coords: (-75.1589597, 39.9787144)
Candidate 1154: Distance = 1146.32m | Coords: (-75.1934836, 39.9530964)
Candidate 1169: Distance = 2318.13m | Coords: (-75.163903, 39.9625073)
Candidate 1253: Distance = 4379.80m | Coords: (-75.1443862, 39.9613397)
Candidate 1693: Distance = 2799.83m | Coords: (-75.2082755, 39.9581761)
Candidate 1811: Distance = 2312.30m | Coords: (-75.1632279, 39.9491182)
Candidate 2076: Distance = 1813.24m | Coords: (-75.1669846, 39.9537042)


In [59]:
import pandas as pd
from shapely.geometry import Point

# 1. Setup the Test Data from your JSON
test_instruction = "Meet me at Dunkin's fast food south of Market street. A theatre is southeast and Books-A-Million is on the opposite block, northeast."
gold_lat, gold_lon = 39.9512020308, -75.156534795
start_lat, start_lon = 39.9479852, -75.166819

print(f"🎯 Target Noun: Dunkin")
print(f"📍 Start: ({start_lon}, {start_lat})")
print(f"🏁 Gold Goal: ({gold_lon}, {gold_lat})")

# 2. Check the POI File for the Target
dunkin_candidates = poi_df[poi_df['name'].str.contains("Dunkin", case=False, na=False)]
print(f"\n🔍 Found {len(dunkin_candidates)} Dunkin candidates.")

# 3. Check for the Anchors (This is where the failure likely is)
anchors = ["Market Street", "theatre", "Books-A-Million"]
for anchor in anchors:
    found = poi_df[poi_df['name'].str.contains(anchor, case=False, na=False)]
    print(f"⚓ Anchor '{anchor}': {'✅ Found' if not found.empty else '❌ MISSING'} ({len(found)} matches)")

# 4. Spatial Verification
def get_dist(lon1, lat1, lon2, lat2):
    return Point(lon1, lat1).distance(Point(lon2, lat2)) * 111000

# Find the closest Dunkin to the GOLD goal point
dunkin_candidates['dist_to_gold'] = dunkin_candidates.apply(
    lambda r: get_dist(gold_lon, gold_lat, r['centroid'].x, r['centroid'].y), axis=1
)

best_match = dunkin_candidates.nsmallest(1, 'dist_to_gold')
print(f"\n🏆 Closest Dunkin to Gold Goal is {best_match['dist_to_gold'].values[0]:.2f}m away.")

🎯 Target Noun: Dunkin
📍 Start: (-75.166819, 39.9479852)
🏁 Gold Goal: (-75.156534795, 39.9512020308)

🔍 Found 31 Dunkin candidates.
⚓ Anchor 'Market Street': ✅ Found (17 matches)
⚓ Anchor 'theatre': ✅ Found (14 matches)
⚓ Anchor 'Books-A-Million': ✅ Found (1 matches)

🏆 Closest Dunkin to Gold Goal is 0.00m away.


The "Wedge Validation" Unit Test after updating utils.py

In [ ]:
import src.utils as utils
import config
import math

# 1. Setup the exact coords from the "Ambiguous" Dunkin row
start_lat, start_lon = 39.9479852, -75.166819
gold_lat, gold_lon = 39.9512020308, -75.156534795  # This Dunkin is actually NE of start

# 2. Test the specific "South of Market Street" logic
# Let's assume a point on Market Street near the target
market_lat, market_lon = 39.953, -75.156 

print(f"Current Config Wedge: {config.DIRECTIONAL_WEDGE_DEGREES}°")

# 3. Check Direction from Market Street to Dunkin
direction = utils.get_dominant_direction(market_lat, market_lon, gold_lat, gold_lon)
bearing = utils.calculate_bearing(market_lat, market_lon, gold_lat, gold_lon)

print(f"\n--- Directional Test ---")
print(f"Origin (Market St): ({market_lat}, {market_lon})")
print(f"Target (Dunkin):    ({gold_lat}, {gold_lon})")
print(f"Calculated Bearing: {bearing:.2f}°")
print(f"Oracle says Direction is: **{direction}**")

# 4. Check Proximity Logic
# Use the radius we set in config
radius_to_test = getattr(config, 'SUCCESS_RADIUS', 150) # Default to 150m if not set
dist = utils.get_geodesic_dist_raw(start_lat, start_lon, gold_lat, gold_lon)
is_near = dist <= radius_to_test

print(f"\n--- Proximity Test ---")
print(f"Distance Start -> Dunkin: {dist:.2f}m")
print(f"Success Radius: {radius_to_test}m")
print(f"Would Oracle count this as 'At' the landmark? {'✅ YES' if is_near else '❌ NO'}")

Current Config Wedge: 90°

--- Directional Test ---
Origin (Market St): (39.953, -75.156)
Target (Dunkin):    (39.9512020308, -75.156534795)
Calculated Bearing: 192.84°
Oracle says Direction is: **S**

--- Proximity Test ---
Distance Start -> Dunkin: 948.66m
Success Radius: 150m
Would Oracle count this as 'At' the landmark? ❌ NO


In [62]:
import src.utils as utils
import config
import math

# 1. Coordinate Setup (Philly University City area)
# Start at a street corner
start_lat, start_lon = 39.9522, -75.1932 
# Anchor: A bookstore (Books-A-Million simulation)
anchor_lat, anchor_lon = 39.9535, -75.1910 
# Goal: A target that is NORTHEAST of the anchor
goal_lat, goal_lon = 39.9550, -75.1890 

print(f"--- Config Check ---")
print(f"City: {config.CURRENT_CITY}")
print(f"Wedge: {config.DIRECTIONAL_WEDGE_DEGREES}°")
print(f"Horizon: {config.GLOBAL_SEARCH_HORIZON_METERS}m")
print(f"Success Radius: {config.get_success_radius()}m")

# 2. Test Direction: Goal relative to Anchor
# Instruction: "The target is Northeast of the bookstore"
bearing = utils.calculate_bearing(anchor_lat, anchor_lon, goal_lat, goal_lon)
direction = utils.get_dominant_direction(anchor_lat, anchor_lon, goal_lat, goal_lon)

print(f"\n--- Logic Test: 'Northeast of Bookstore' ---")
print(f"Bearing from Anchor to Goal: {bearing:.2f}°")
print(f"Oracle interprets this as: **{direction}**")

# 3. Test Horizon: Can the Oracle even 'see' the goal from the Start?
dist_to_goal = utils.get_geodesic_dist_raw(start_lat, start_lon, goal_lat, goal_lon)
can_see = dist_to_goal <= config.GLOBAL_SEARCH_HORIZON_METERS

print(f"\n--- Horizon Test ---")
print(f"Distance Start -> Goal: {dist_to_goal:.2f}m")
print(f"Within 1500m Horizon? {'✅ YES' if can_see else '❌ NO'}")

# 4. Success Test: If the Agent reaches these coords, is it a 'Success'?
# We simulate the agent being exactly at the goal coords
is_success = dist_to_goal <= config.get_success_radius() # This should be False (it's the travel distance)
# We simulate the agent being 90m away from the goal
is_at_goal = 90.0 <= config.get_success_radius()

print(f"\n--- Success Radius Test (100m) ---")
print(f"Agent 90m from goal. Is it a success? {'✅ YES' if is_at_goal else '❌ NO'}")

--- Config Check ---
City: manhattan
Wedge: 90°
Horizon: 1500m
Success Radius: 80m

--- Logic Test: 'Northeast of Bookstore' ---
Bearing from Anchor to Goal: 45.62°
Oracle interprets this as: **E**

--- Horizon Test ---
Distance Start -> Goal: 474.83m
Within 1500m Horizon? ✅ YES

--- Success Radius Test (100m) ---
Agent 90m from goal. Is it a success? ❌ NO
